<a href="https://colab.research.google.com/github/HussainSheikh45/Fee-management-system/blob/main/Fee_management_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Create the project folder structure
import os
os.makedirs("am-anglo-fee-system/data", exist_ok=True)
os.chdir("am-anglo-fee-system")
print("Folder ready:", os.getcwd())

Folder ready: /content/am-anglo-fee-system


In [4]:
"""
database.py — AM Anglo School Fee Management System
====================================================
Single source of truth for all database operations.
Handles schema creation, CRUD, and connection lifecycle.

Design principles:
  - All queries parameterized (zero SQL injection surface)
  - Context-manager connections (guaranteed cleanup)
  - Separate fee_structure table (fee changes don't corrupt history)
  - Soft deletes on students (retain payment history even after a student leaves)
  - Indexed foreign keys for query performance at scale
"""

import sqlite3
import logging
from pathlib import Path
from datetime import datetime, date
from typing import Optional

# ── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

# ── Config ────────────────────────────────────────────────────────────────────
from pathlib import Path

# Works in Colab, Jupyter, Streamlit, and local Python
DB_DIR = Path("data")
DB_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "school.db"


# ── Schema ────────────────────────────────────────────────────────────────────
SCHEMA = """
-- Students: core identity and contact
CREATE TABLE IF NOT EXISTS students (
    student_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    student_name    TEXT    NOT NULL,
    class           TEXT    NOT NULL,
    guardian_name   TEXT    NOT NULL,
    whatsapp_number TEXT    NOT NULL,
    is_active       INTEGER NOT NULL DEFAULT 1,        -- soft delete flag
    created_at      TEXT    NOT NULL DEFAULT (datetime('now', 'localtime')),
    updated_at      TEXT    NOT NULL DEFAULT (datetime('now', 'localtime'))
);

-- Fee structure: monthly fee per student, tracked over time
-- Separate table so mid-year fee changes don't alter payment history
CREATE TABLE IF NOT EXISTS fee_structure (
    fee_id          INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id      INTEGER NOT NULL REFERENCES students(student_id) ON DELETE CASCADE,
    monthly_fee     REAL    NOT NULL CHECK (monthly_fee > 0),
    effective_from  TEXT    NOT NULL,                  -- YYYY-MM-DD
    effective_to    TEXT,                              -- NULL = currently active
    created_at      TEXT    NOT NULL DEFAULT (datetime('now', 'localtime'))
);

-- Payments: immutable transaction ledger
-- Records are never deleted — only status changes are allowed
CREATE TABLE IF NOT EXISTS payments (
    payment_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id      INTEGER NOT NULL REFERENCES students(student_id) ON DELETE CASCADE,
    amount_paid     REAL    NOT NULL CHECK (amount_paid > 0),
    payment_date    TEXT    NOT NULL,                  -- YYYY-MM-DD
    month_covered   TEXT    NOT NULL,                  -- YYYY-MM  (e.g. 2025-01)
    payment_status  TEXT    NOT NULL DEFAULT 'Paid'
                            CHECK (payment_status IN ('Paid', 'Partial', 'Waived')),
    notes           TEXT,
    recorded_by     TEXT    NOT NULL DEFAULT 'System',
    created_at      TEXT    NOT NULL DEFAULT (datetime('now', 'localtime'))
);

-- Indexes for common query patterns
CREATE INDEX IF NOT EXISTS idx_payments_student   ON payments(student_id);
CREATE INDEX IF NOT EXISTS idx_payments_month     ON payments(month_covered);
CREATE INDEX IF NOT EXISTS idx_payments_date      ON payments(payment_date);
CREATE INDEX IF NOT EXISTS idx_fee_student        ON fee_structure(student_id);
CREATE INDEX IF NOT EXISTS idx_students_class     ON students(class);
CREATE INDEX IF NOT EXISTS idx_students_active    ON students(is_active);
"""


# ── Connection helper ─────────────────────────────────────────────────────────
def get_connection() -> sqlite3.Connection:
    """
    Return a configured SQLite connection.
    - Row factory enables dict-style access: row['column_name']
    - Foreign key enforcement is OFF by default in SQLite; we turn it ON.
    - WAL journal mode gives better concurrent read performance.
    """
    DB_DIR.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(str(DB_PATH))
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("PRAGMA journal_mode = WAL")
    return conn


# ── Schema initialisation ─────────────────────────────────────────────────────
def init_db() -> None:
    """Create all tables and indexes if they do not already exist."""
    try:
        with get_connection() as conn:
            conn.executescript(SCHEMA)
        logger.info("Database initialised at %s", DB_PATH)
    except sqlite3.Error as e:
        logger.error("Failed to initialise database: %s", e)
        raise


# ══════════════════════════════════════════════════════════════════════════════
#  STUDENT CRUD
# ══════════════════════════════════════════════════════════════════════════════

def add_student(
    student_name: str,
    class_name: str,
    guardian_name: str,
    whatsapp_number: str,
    monthly_fee: float,
    fee_effective_from: Optional[str] = None,
) -> int:
    """
    Insert a new student and their initial fee structure in a single transaction.

    Returns the new student_id.
    Raises ValueError for bad inputs; sqlite3.Error for DB failures.
    """
    if not all([student_name, class_name, guardian_name, whatsapp_number]):
        raise ValueError("student_name, class_name, guardian_name, and whatsapp_number are required.")
    if monthly_fee <= 0:
        raise ValueError("monthly_fee must be greater than zero.")

    effective_from = fee_effective_from or date.today().strftime("%Y-%m-%d")

    try:
        with get_connection() as conn:
            cursor = conn.execute(
                """
                INSERT INTO students (student_name, class, guardian_name, whatsapp_number)
                VALUES (?, ?, ?, ?)
                """,
                (student_name.strip(), class_name.strip(),
                 guardian_name.strip(), whatsapp_number.strip()),
            )
            student_id = cursor.lastrowid

            conn.execute(
                """
                INSERT INTO fee_structure (student_id, monthly_fee, effective_from)
                VALUES (?, ?, ?)
                """,
                (student_id, monthly_fee, effective_from),
            )

        logger.info("Added student '%s' (ID %d) with fee %.2f", student_name, student_id, monthly_fee)
        return student_id

    except sqlite3.IntegrityError as e:
        logger.error("Integrity error adding student '%s': %s", student_name, e)
        raise
    except sqlite3.Error as e:
        logger.error("Database error adding student '%s': %s", student_name, e)
        raise


def get_all_students(include_inactive: bool = False) -> list[dict]:
    """
    Return all students joined with their current monthly fee.
    By default returns only active students.
    """
    active_filter = "" if include_inactive else "WHERE s.is_active = 1"

    query = f"""
        SELECT
            s.student_id,
            s.student_name,
            s.class,
            s.guardian_name,
            s.whatsapp_number,
            s.is_active,
            s.created_at,
            COALESCE(fs.monthly_fee, 0) AS monthly_fee
        FROM students s
        LEFT JOIN fee_structure fs
            ON fs.student_id = s.student_id
            AND fs.effective_to IS NULL          -- only the currently active fee
        {active_filter}
        ORDER BY s.class, s.student_name
    """
    try:
        with get_connection() as conn:
            rows = conn.execute(query).fetchall()
            return [dict(row) for row in rows]
    except sqlite3.Error as e:
        logger.error("Error fetching students: %s", e)
        raise


def get_student_by_id(student_id: int) -> Optional[dict]:
    """Return a single student record with current fee, or None if not found."""
    query = """
        SELECT
            s.student_id, s.student_name, s.class, s.guardian_name,
            s.whatsapp_number, s.is_active, s.created_at,
            COALESCE(fs.monthly_fee, 0) AS monthly_fee
        FROM students s
        LEFT JOIN fee_structure fs
            ON fs.student_id = s.student_id AND fs.effective_to IS NULL
        WHERE s.student_id = ?
    """
    try:
        with get_connection() as conn:
            row = conn.execute(query, (student_id,)).fetchone()
            return dict(row) if row else None
    except sqlite3.Error as e:
        logger.error("Error fetching student ID %d: %s", student_id, e)
        raise


def update_student(
    student_id: int,
    student_name: Optional[str] = None,
    class_name: Optional[str] = None,
    guardian_name: Optional[str] = None,
    whatsapp_number: Optional[str] = None,
) -> bool:
    """
    Update student contact/identity fields.
    Only updates fields that are explicitly passed.
    Returns True if a row was modified.
    """
    fields, params = [], []
    if student_name:
        fields.append("student_name = ?"); params.append(student_name.strip())
    if class_name:
        fields.append("class = ?"); params.append(class_name.strip())
    if guardian_name:
        fields.append("guardian_name = ?"); params.append(guardian_name.strip())
    if whatsapp_number:
        fields.append("whatsapp_number = ?"); params.append(whatsapp_number.strip())

    if not fields:
        logger.warning("update_student called with no fields to update.")
        return False

    fields.append("updated_at = datetime('now', 'localtime')")
    params.append(student_id)

    try:
        with get_connection() as conn:
            cursor = conn.execute(
                f"UPDATE students SET {', '.join(fields)} WHERE student_id = ?",
                params,
            )
            modified = cursor.rowcount > 0
        if modified:
            logger.info("Updated student ID %d", student_id)
        return modified
    except sqlite3.Error as e:
        logger.error("Error updating student ID %d: %s", student_id, e)
        raise


def update_student_fee(
    student_id: int,
    new_monthly_fee: float,
    effective_from: Optional[str] = None,
) -> None:
    """
    Change a student's monthly fee without touching historical records.
    Closes the current active fee_structure row and inserts a new one.
    """
    if new_monthly_fee <= 0:
        raise ValueError("new_monthly_fee must be greater than zero.")

    today = effective_from or date.today().strftime("%Y-%m-%d")

    try:
        with get_connection() as conn:
            # Close out the current active fee record
            conn.execute(
                """
                UPDATE fee_structure
                SET effective_to = ?
                WHERE student_id = ? AND effective_to IS NULL
                """,
                (today, student_id),
            )
            # Insert new active fee
            conn.execute(
                """
                INSERT INTO fee_structure (student_id, monthly_fee, effective_from)
                VALUES (?, ?, ?)
                """,
                (student_id, new_monthly_fee, today),
            )
        logger.info(
            "Updated fee for student ID %d to %.2f effective %s",
            student_id, new_monthly_fee, today,
        )
    except sqlite3.Error as e:
        logger.error("Error updating fee for student ID %d: %s", student_id, e)
        raise


def deactivate_student(student_id: int) -> bool:
    """
    Soft-delete a student (sets is_active = 0).
    Payment history is fully preserved.
    Returns True if the student was found and deactivated.
    """
    try:
        with get_connection() as conn:
            cursor = conn.execute(
                "UPDATE students SET is_active = 0, updated_at = datetime('now', 'localtime') WHERE student_id = ?",
                (student_id,),
            )
            modified = cursor.rowcount > 0
        if modified:
            logger.info("Deactivated student ID %d", student_id)
        return modified
    except sqlite3.Error as e:
        logger.error("Error deactivating student ID %d: %s", student_id, e)
        raise


# ══════════════════════════════════════════════════════════════════════════════
#  PAYMENT CRUD
# ══════════════════════════════════════════════════════════════════════════════

def record_payment(
    student_id: int,
    amount_paid: float,
    payment_date: str,
    month_covered: str,
    payment_status: str = "Paid",
    notes: Optional[str] = None,
    recorded_by: str = "System",
) -> int:
    """
    Insert a payment record.

    Args:
        student_id:     Must exist in the students table.
        amount_paid:    Positive float.
        payment_date:   ISO format YYYY-MM-DD.
        month_covered:  YYYY-MM  (e.g. '2025-01').
        payment_status: 'Paid' | 'Partial' | 'Waived'
        notes:          Optional free-text note.
        recorded_by:    Staff name or 'System' for imports.

    Returns the new payment_id.
    """
    if amount_paid <= 0:
        raise ValueError("amount_paid must be greater than zero.")
    if payment_status not in ("Paid", "Partial", "Waived"):
        raise ValueError("payment_status must be 'Paid', 'Partial', or 'Waived'.")

    # Validate date formats
    try:
        datetime.strptime(payment_date, "%Y-%m-%d")
        datetime.strptime(month_covered, "%Y-%m")
    except ValueError:
        raise ValueError("payment_date must be YYYY-MM-DD and month_covered must be YYYY-MM.")

    try:
        with get_connection() as conn:
            cursor = conn.execute(
                """
                INSERT INTO payments
                    (student_id, amount_paid, payment_date, month_covered,
                     payment_status, notes, recorded_by)
                VALUES (?, ?, ?, ?, ?, ?, ?)
                """,
                (student_id, amount_paid, payment_date,
                 month_covered, payment_status, notes, recorded_by),
            )
            payment_id = cursor.lastrowid
        logger.info(
            "Recorded payment ID %d for student ID %d — %.2f covering %s",
            payment_id, student_id, amount_paid, month_covered,
        )
        return payment_id
    except sqlite3.IntegrityError as e:
        logger.error("Integrity error recording payment for student ID %d: %s", student_id, e)
        raise
    except sqlite3.Error as e:
        logger.error("Database error recording payment: %s", e)
        raise


def get_payments_by_student(student_id: int) -> list[dict]:
    """Return all payment records for a student, newest first."""
    try:
        with get_connection() as conn:
            rows = conn.execute(
                """
                SELECT * FROM payments
                WHERE student_id = ?
                ORDER BY payment_date DESC, created_at DESC
                """,
                (student_id,),
            ).fetchall()
            return [dict(row) for row in rows]
    except sqlite3.Error as e:
        logger.error("Error fetching payments for student ID %d: %s", student_id, e)
        raise


def get_payments_by_month(month_covered: str) -> list[dict]:
    """
    Return all payments for a given month (YYYY-MM), joined with student info.
    Used by report_generator.py for monthly collection reports.
    """
    try:
        with get_connection() as conn:
            rows = conn.execute(
                """
                SELECT
                    p.*,
                    s.student_name,
                    s.class,
                    s.guardian_name,
                    s.whatsapp_number
                FROM payments p
                JOIN students s ON s.student_id = p.student_id
                WHERE p.month_covered = ?
                ORDER BY s.class, s.student_name
                """,
                (month_covered,),
            ).fetchall()
            return [dict(row) for row in rows]
    except sqlite3.Error as e:
        logger.error("Error fetching payments for month %s: %s", month_covered, e)
        raise


def get_all_payments() -> list[dict]:
    """Return every payment record joined with student data. Used for bulk reports."""
    try:
        with get_connection() as conn:
            rows = conn.execute(
                """
                SELECT
                    p.*,
                    s.student_name,
                    s.class,
                    s.guardian_name,
                    s.whatsapp_number
                FROM payments p
                JOIN students s ON s.student_id = p.student_id
                ORDER BY p.payment_date DESC
                """
            ).fetchall()
            return [dict(row) for row in rows]
    except sqlite3.Error as e:
        logger.error("Error fetching all payments: %s", e)
        raise


def delete_payment(payment_id: int) -> bool:
    """
    Hard-delete a payment record.
    Use sparingly — only for genuine data-entry errors.
    Prefer recording a corrective 'Waived' entry instead.
    Returns True if a record was deleted.
    """
    try:
        with get_connection() as conn:
            cursor = conn.execute(
                "DELETE FROM payments WHERE payment_id = ?",
                (payment_id,),
            )
            deleted = cursor.rowcount > 0
        if deleted:
            logger.warning("DELETED payment ID %d — this action is irreversible.", payment_id)
        return deleted
    except sqlite3.Error as e:
        logger.error("Error deleting payment ID %d: %s", payment_id, e)
        raise


# ══════════════════════════════════════════════════════════════════════════════
#  UTILITY QUERIES
# ══════════════════════════════════════════════════════════════════════════════

def get_all_classes() -> list[str]:
    """Return a sorted list of distinct class names for dropdowns/filters."""
    try:
        with get_connection() as conn:
            rows = conn.execute(
                "SELECT DISTINCT class FROM students WHERE is_active = 1 ORDER BY class"
            ).fetchall()
            return [row["class"] for row in rows]
    except sqlite3.Error as e:
        logger.error("Error fetching class list: %s", e)
        raise


def get_db_summary() -> dict:
    """
    Return a quick health-check summary of the database.
    Shown on the Streamlit dashboard homepage.
    """
    try:
        with get_connection() as conn:
            student_count = conn.execute(
                "SELECT COUNT(*) FROM students WHERE is_active = 1"
            ).fetchone()[0]

            payment_count = conn.execute(
                "SELECT COUNT(*) FROM payments"
            ).fetchone()[0]

            total_collected = conn.execute(
                "SELECT COALESCE(SUM(amount_paid), 0) FROM payments"
            ).fetchone()[0]

            class_count = conn.execute(
                "SELECT COUNT(DISTINCT class) FROM students WHERE is_active = 1"
            ).fetchone()[0]

        return {
            "active_students": student_count,
            "total_payments": payment_count,
            "total_collected": round(total_collected, 2),
            "class_count": class_count,
        }
    except sqlite3.Error as e:
        logger.error("Error fetching DB summary: %s", e)
        raise


def bulk_insert_students(students: list[dict]) -> tuple[int, int]:
    """
    Bulk-import students from a list of dicts (used by Excel importer).
    Each dict must have: student_name, class, guardian_name, whatsapp_number, monthly_fee.
    Skips rows with missing required fields.

    Returns (success_count, skip_count).
    """
    success, skipped = 0, 0
    required_keys = {"student_name", "class", "guardian_name", "whatsapp_number", "monthly_fee"}

    for row in students:
        missing = required_keys - set(row.keys())
        if missing or not row.get("student_name"):
            logger.warning("Skipping row — missing fields: %s", missing)
            skipped += 1
            continue
        try:
            add_student(
                student_name=str(row["student_name"]),
                class_name=str(row["class"]),
                guardian_name=str(row["guardian_name"]),
                whatsapp_number=str(row["whatsapp_number"]),
                monthly_fee=float(row["monthly_fee"]),
            )
            success += 1
        except Exception as e:
            logger.warning("Skipping row for '%s': %s", row.get("student_name"), e)
            skipped += 1

    logger.info("Bulk insert complete — %d added, %d skipped", success, skipped)
    return success, skipped


# ── Entry point (sanity check) ────────────────────────────────────────────────
if __name__ == "__main__":
    init_db()
    summary = get_db_summary()
    print("\n── AM Anglo School DB ──────────────────")
    for k, v in summary.items():
        print(f"  {k:<22} {v}")
    print("────────────────────────────────────────\n")


── AM Anglo School DB ──────────────────
  active_students        0
  total_payments         0
  total_collected        0
  class_count            0
────────────────────────────────────────



In [5]:
# Initialize the database
init_db()

# Add a test student
id1 = add_student("Ali Hassan", "Class 5", "Mr Hassan", "923001234567", 3500)
id2 = add_student("Sara Khan", "Class 6", "Mrs Khan", "923007654321", 4000)

# Record a payment
record_payment(id1, 3500, "2025-01-05", "2025-01", "Paid", recorded_by="Admin")
record_payment(id2, 2000, "2025-01-10", "2025-01", "Partial", notes="Balance pending")

# Check summary
summary = get_db_summary()
print(summary)

# List all students
students = get_all_students()
for s in students:
    print(s['student_name'], "|", s['class'], "| Fee:", s['monthly_fee'])

{'active_students': 2, 'total_payments': 2, 'total_collected': 5500.0, 'class_count': 2}
Ali Hassan | Class 5 | Fee: 3500.0
Sara Khan | Class 6 | Fee: 4000.0
